# Monetizing Quantum Compute with x402 HTTP Payments

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ionq-samples/getting-started/blob/main/x402_quantum_payments.ipynb)

The [x402 protocol](https://www.x402.org/) brings the HTTP `402 Payment Required` status code to life — letting any API gate access behind stablecoin micropayments, with no subscription or billing account needed.

This notebook walks through the full protocol flow:

1. **Public discovery** — find payment-gated endpoints and their pricing
2. **Unpaid 402 challenge** — inspect the payment requirement before any money moves
3. **Wallet approval** — user/agent explicitly authorizes the payment
4. **Signed payment retry** — resend with cryptographic payment proof
5. **Job execution** — gateway verifies payment on-chain, forwards to IonQ API

## Why This Matters for Quantum

Quantum compute has a natural fit with micropayments:
- **Per-shot pricing** already mirrors per-request billing
- **AI agents** need autonomous compute access without human billing setup
- **Low-friction onboarding** — run a circuit with just a wallet, no account creation
- **Cross-border access** — USDC payments work globally without merchant accounts

In [ ]:
# Install dependencies
!pip install requests qiskit qiskit-ionq

## Configuration

### Gateway trust boundary

An x402 gateway sits between your client and the upstream API. This means the
gateway operator can see any headers you send through it — **including API keys**.

Before sending real credentials through any gateway, verify:
- **Who operates it** — is it the API provider, a known infrastructure partner, or a third party?
- **What it can see** — the gateway terminates TLS and can read request/response bodies
- **Whether it's contractually authorized** — does the upstream API provider endorse this gateway?

⚠️ **Do not send real IonQ API keys through a third-party gateway unless you trust
the operator and have verified the trust relationship.** For this notebook, we use
the simulator with a placeholder gateway URL. Replace it with a gateway you trust
before using real credentials or QPU targets.

Gateway options:
- **Self-host** using the [Coinbase x402 reference implementation](https://github.com/coinbase/x402)
- **Provider-operated** gateway run by the quantum compute provider (ideal — no third-party trust needed)
- **Third-party hosted** (e.g., `gateway.spraay.app`) — verify trust before sending API keys

In [ ]:
import os
import json
import requests
from datetime import datetime

# ── Gateway URL ────────────────────────────────────────────
# Replace with a gateway you trust. See trust boundary notes above.
X402_GATEWAY = os.environ.get("X402_GATEWAY_URL", "YOUR_X402_GATEWAY_URL")

# ── IonQ API Key ──────────────────────────────────────────
# Only send this through a gateway you trust (see above).
IONQ_API_KEY = os.environ.get("IONQ_API_KEY", "your_ionq_api_key")

if X402_GATEWAY == "YOUR_X402_GATEWAY_URL":
    print("⚠️  Set X402_GATEWAY_URL to a gateway you trust before running payment cells.")
    print("   Self-host: https://github.com/coinbase/x402")
    print("   Or use a provider-operated gateway.")
else:
    print(f"Gateway: {X402_GATEWAY}")
    print(f"Protocol: x402 (HTTP 402 Payment Required)")
    print(f"Settlement: USDC on Base L2")

## Step 1: Public Discovery

x402 gateways advertise payment-gated endpoints at well-known paths.
Common discovery endpoints across implementations include:

| Path | Convention |
|------|------------|
| `/.well-known/x402` | Coinbase reference implementation |
| `/.well-known/x402.json` | JSON-specific variant |
| `/.well-known/x402-manifest` | Extended manifest format |

The discovery step is read-only — no payment, no credentials needed.
This lets agents inspect what's available and at what price before committing.

In [ ]:
# Common x402 discovery paths across gateway implementations
DISCOVERY_PATHS = [
    "/.well-known/x402",
    "/.well-known/x402.json",
    "/.well-known/x402-manifest",
]


def discover_x402_endpoints(gateway_url, keyword=None):
    """
    Probe a gateway's well-known paths for endpoint discovery.

    Tries common x402 discovery paths and returns the first
    successful response. No credentials or payment needed.
    """
    for path in DISCOVERY_PATHS:
        try:
            resp = requests.get(
                f"{gateway_url}{path}",
                headers={"Accept": "application/json"},
                timeout=10,
            )
            if resp.status_code == 200:
                data = resp.json()
                endpoints = data.get("endpoints", data.get("resources", []))
                print(f"✅ Discovery succeeded at {path}")
                print(f"   Found {len(endpoints)} endpoint(s)\n")
                if keyword:
                    endpoints = [
                        ep for ep in endpoints
                        if keyword.lower() in json.dumps(ep).lower()
                    ]
                return {"discovery_path": path, "endpoints": endpoints}
        except requests.RequestException:
            continue

    print("❌ No discovery endpoint responded.")
    print(f"   Tried: {', '.join(DISCOVERY_PATHS)}")
    return None


# Run discovery (no payment, no credentials)
if X402_GATEWAY != "YOUR_X402_GATEWAY_URL":
    discovery = discover_x402_endpoints(X402_GATEWAY, keyword="inference")
    if discovery:
        for ep in discovery["endpoints"][:5]:
            print(f"  {ep.get('method', 'POST')} {ep.get('path', '')}")
            print(f"    Price: {ep.get('price', 'N/A')} USDC")
            print(f"    {ep.get('description', '')}")
            print()
else:
    print("⏭️  Skipped — set X402_GATEWAY_URL first.")

## Step 2: Unpaid 402 Challenge Inspection

Before any money moves, send an **unpaid request** to the payment-gated endpoint.
The gateway returns a `402 Payment Required` response containing the exact payment
requirement. This is a smoke test — verify the challenge shape before proceeding.

A well-formed 402 challenge should include:

| Field | Description |
|-------|-------------|
| `amount` / `price` | Cost in the settlement asset |
| `asset` / `currency` | Settlement token (e.g., USDC) |
| `network` | Settlement chain (e.g., base, ethereum) |
| `payTo` | Recipient address for payment |
| `resource` | The endpoint being gated |
| `expiry` | How long the quote is valid |

Also inspect response headers for `Cache-Control` and CORS headers.

In [ ]:
def inspect_402_challenge(gateway_url, path, payload=None):
    """
    Send an unpaid request and inspect the 402 challenge.

    This is a safe, no-cost validation step. The gateway returns
    the payment requirement without executing anything or moving
    any funds. Use this to verify the gateway is well-behaved
    before authorizing payment.

    No API keys are sent in this step.
    """
    try:
        resp = requests.post(
            f"{gateway_url}{path}",
            json=payload or {},
            headers={"Content-Type": "application/json"},
            timeout=15,
        )
    except requests.RequestException as e:
        print(f"❌ Request failed: {e}")
        return None

    print(f"HTTP {resp.status_code}")
    print()

    if resp.status_code != 402:
        print(f"⚠️  Expected 402, got {resp.status_code}")
        print(f"   Body: {resp.text[:300]}")
        return None

    # Parse the 402 challenge body
    try:
        challenge = resp.json()
    except ValueError:
        print("⚠️  402 response is not valid JSON")
        return None

    # Validate expected fields
    expected_fields = ["price", "amount", "currency", "asset",
                       "network", "payTo", "resource", "expiry"]
    print("── 402 Challenge Body ──────────────────────────")
    for key, value in challenge.items():
        marker = "✅" if key in expected_fields else "  "
        print(f"  {marker} {key}: {value}")

    # Check which expected fields are present
    present = [f for f in expected_fields if f in challenge]
    missing = [f for f in expected_fields if f not in challenge]
    print(f"\n  Present: {', '.join(present) if present else 'none'}")
    if missing:
        # Some fields may use alternate names, so this is informational
        print(f"  Not found: {', '.join(missing)} (may use alternate field names)")

    # Inspect response headers
    print("\n── Response Headers (relevant) ────────────────")
    for header in ["Cache-Control", "Access-Control-Allow-Origin",
                   "Access-Control-Allow-Headers", "X-Payment-Required"]:
        val = resp.headers.get(header)
        if val:
            print(f"  {header}: {val}")

    print("────────────────────────────────────────────────")
    return challenge


# Smoke test — no payment, no API key
if X402_GATEWAY != "YOUR_X402_GATEWAY_URL":
    print("Sending unpaid request to inspect 402 challenge...\n")
    challenge = inspect_402_challenge(
        X402_GATEWAY,
        "/v0.1/ai-inference",
        payload={"target": "simulator", "shots": 100, "input": {}}
    )
else:
    print("⏭️  Skipped — set X402_GATEWAY_URL first.")

## Step 3: Build a Quantum Circuit

Before involving payments, build the circuit payload locally.

In [ ]:
# Bell State circuit in IonQ's native JSON format
bell_state = {
    "format": "ionq.circuit.v0",
    "qubits": 2,
    "circuit": [
        {"gate": "h", "target": 0},
        {"gate": "cnot", "control": 0, "target": 1}
    ]
}

print("Circuit: H(q0) → CNOT(q0, q1)")
print("Expected output: |00⟩ and |11⟩ with ~50/50 probability")
print(f"\nPayload:\n{json.dumps(bell_state, indent=2)}")

## Step 4: Signed Payment Submission

The full x402 payment flow has three distinct phases:

```
Phase 1: Unpaid request → 402 challenge (done in Step 2)
Phase 2: Wallet signs a payment authorization covering the quoted amount
Phase 3: Retry with signed proof in X-PAYMENT-PROOF header → 200 + result
```

The code below shows the **complete flow** — but the actual payment signing
(Phase 2) requires a funded wallet and a client library. We mark the signing
step as pseudocode.

### Client libraries for payment signing

| Library | Language | Notes |
|---------|----------|-------|
| [`@coinbase/x402`](https://github.com/coinbase/x402) | TypeScript | Reference implementation |
| [Coinbase AgentKit](https://github.com/coinbase/agentkit) | Python/TS | AI agent wallet with x402 support |
| [Spraay Agent Wallet SDK](https://www.npmjs.com/package/spraay-agent-wallet) | TypeScript | Managed agent wallets on Base |

In [ ]:
def submit_circuit_x402(gateway_url, circuit_json, shots=100, api_key=None):
    """
    Submit a quantum circuit through an x402 gateway.

    Implements the full three-phase protocol flow:

    Phase 1 — Unpaid request: send the job without payment proof.
              Gateway returns 402 with payment requirement.

    Phase 2 — Payment signing (pseudocode): a wallet client library
              signs a USDC authorization covering the quoted amount.
              This step requires a funded wallet and a real client SDK.

    Phase 3 — Paid retry: resend with X-PAYMENT-PROOF header containing
              the signed payment. Gateway verifies on-chain and proxies
              the authenticated request to IonQ.

    ⚠️ API key trust: if api_key is provided, it is sent to the gateway
    in the Authorization header. Only provide this if you trust the
    gateway operator. See the trust boundary notes at the top.
    """
    payload = {
        "target": "simulator",
        "shots": shots,
        "input": circuit_json,
    }

    # ── Phase 1: Unpaid request → 402 challenge ──────────
    print("Phase 1: Sending unpaid request...")
    resp = requests.post(
        f"{gateway_url}/v0.1/ai-inference",
        json=payload,
        headers={"Content-Type": "application/json"},
        timeout=15,
    )

    if resp.status_code != 402:
        print(f"  Unexpected status: {resp.status_code}")
        return None

    challenge = resp.json()
    amount = challenge.get("price") or challenge.get("amount", "?")
    network = challenge.get("network", "?")
    pay_to = challenge.get("payTo", "?")
    print(f"  402 received — {amount} USDC on {network}")
    print(f"  Pay to: {pay_to}")

    # ── Phase 2: Payment signing (pseudocode) ────────────
    #
    # In production, a wallet client library handles this:
    #
    #   from x402_client import sign_payment
    #   payment_proof = sign_payment(
    #       wallet=agent_wallet,
    #       amount=challenge["price"],
    #       asset="USDC",
    #       network=challenge["network"],
    #       pay_to=challenge["payTo"],
    #       resource=challenge.get("resource", "/v0.1/ai-inference"),
    #   )
    #
    # The returned payment_proof is a signed authorization that the
    # gateway can verify on-chain without the client broadcasting
    # a transaction first — the gateway settles it.
    #
    print("\nPhase 2: Payment signing (pseudocode — requires wallet SDK)")
    print("  → In production: wallet signs USDC authorization")
    print("  → Libraries: @coinbase/x402, Coinbase AgentKit, etc.")
    payment_proof = "PSEUDOCODE_PROOF_PLACEHOLDER"

    # ── Phase 3: Paid retry ──────────────────────────────
    print("\nPhase 3: Retrying with signed payment proof...")
    headers = {
        "Content-Type": "application/json",
        "X-PAYMENT-PROOF": payment_proof,
    }
    if api_key:
        # ⚠️ Only include if you trust the gateway operator
        headers["Authorization"] = f"apiKey {api_key}"

    resp = requests.post(
        f"{gateway_url}/v0.1/ai-inference",
        json=payload,
        headers=headers,
        timeout=30,
    )

    if resp.status_code in (200, 201):
        result = resp.json()
        print(f"  ✅ Job submitted — ID: {result.get('id', 'N/A')}")
        print(f"     Status: {result.get('status', 'N/A')}")
        return result
    elif resp.status_code == 402:
        print(f"  ⚠️  Still 402 — payment proof was not accepted")
        print(f"     (Expected with pseudocode placeholder)")
        return {"status": "payment_proof_required", "challenge": challenge}
    else:
        print(f"  Error {resp.status_code}: {resp.text[:200]}")
        return None


# Run the full flow
if X402_GATEWAY != "YOUR_X402_GATEWAY_URL":
    print("── Full x402 Flow: Bell State Circuit ──────────\n")
    result = submit_circuit_x402(
        X402_GATEWAY, bell_state, shots=1000,
        # api_key=IONQ_API_KEY  # uncomment only for trusted gateways
    )
else:
    print("⏭️  Skipped — set X402_GATEWAY_URL first.")

## Step 5: Qiskit Integration

Build circuits with Qiskit, then convert and submit through the x402 flow.

In [ ]:
from qiskit import QuantumCircuit

def qiskit_to_ionq_json(qc):
    """Convert a Qiskit QuantumCircuit to IonQ native JSON."""
    gate_map = {
        'h':  lambda q, _: {"gate": "h", "target": q[0]},
        'x':  lambda q, _: {"gate": "x", "target": q[0]},
        'y':  lambda q, _: {"gate": "y", "target": q[0]},
        'z':  lambda q, _: {"gate": "z", "target": q[0]},
        'cx': lambda q, _: {"gate": "cnot", "control": q[0], "target": q[1]},
        'rz': lambda q, p: {"gate": "rz", "target": q[0], "rotation": p[0]},
        'ry': lambda q, p: {"gate": "ry", "target": q[0], "rotation": p[0]},
    }
    gates = []
    for inst in qc.data:
        name = inst.operation.name
        if name in gate_map:
            qubits = [qc.find_bit(q).index for q in inst.qubits]
            gates.append(gate_map[name](qubits, inst.operation.params))
    return {"format": "ionq.circuit.v0", "qubits": qc.num_qubits, "circuit": gates}


# Build a 3-qubit GHZ state
ghz = QuantumCircuit(3)
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)

print(ghz.draw(output='text'))
print()

ionq_payload = qiskit_to_ionq_json(ghz)
print(f"IonQ JSON:\n{json.dumps(ionq_payload, indent=2)}")

# Submit via x402 (same flow as Bell state)
if X402_GATEWAY != "YOUR_X402_GATEWAY_URL":
    print("\n── Full x402 Flow: GHZ Circuit ─────────────────\n")
    result = submit_circuit_x402(X402_GATEWAY, ionq_payload, shots=1000)
else:
    print("\n⏭️  Skipped payment flow — set X402_GATEWAY_URL first.")

## Adding x402 to Your Own Quantum API

If you're a quantum compute provider, here's the minimal server-side pattern.
x402 is **additive** — it doesn't replace existing API key auth, it runs alongside it.

```python
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route('/v1/jobs', methods=['POST'])
def submit_job():
    # If no payment header, use existing auth flow (unchanged)
    if 'X-PAYMENT-PROOF' not in request.headers:
        if 'Authorization' in request.headers:
            return existing_api_key_flow(request)
        # No auth at all — return 402 challenge
        return jsonify({
            'price': '0.01',
            'currency': 'USDC',
            'network': 'base',
            'payTo': '0xYourReceiverAddress',
            'resource': '/v1/jobs',
            'expiry': 300,  # seconds
        }), 402

    # Verify the signed payment proof on-chain
    proof = request.headers['X-PAYMENT-PROOF']
    if verify_usdc_payment(proof, expected_amount='0.01'):
        return process_quantum_job(request)
    else:
        return jsonify({'error': 'payment_verification_failed'}), 402
```

Key points:
- Existing API key customers are **unaffected** — the x402 path only activates when payment headers are present
- The 402 challenge includes `resource` and `expiry` so clients know exactly what they're paying for
- Payment verification happens on-chain (Base L2) — no credit card processor needed

## Resources

| Resource | Link |
|----------|------|
| x402 Protocol Specification | [x402.org](https://www.x402.org/) |
| Coinbase x402 Reference (server + client) | [github.com/coinbase/x402](https://github.com/coinbase/x402) |
| Coinbase AgentKit (wallet + x402 signing) | [github.com/coinbase/agentkit](https://github.com/coinbase/agentkit) |
| IonQ API Reference | [docs.ionq.com](https://docs.ionq.com/api-reference/v0.2/) |